# Jira Tickets — Incremental Update & Country Export

This notebook keeps the stored Jira tickets up to date and produces the two
deliverables from a single, consistent country resolution:

1. **Load** the previously pulled tickets from `data/jira_issues.json`.
2. **Pull** only the last `LOOKBACK_DAYS` days of tickets from Jira.
3. **Merge** them into the store by *Ticket Key* (upsert: recent tickets replace
   their older stored version, brand-new keys are added, everything else is kept).
4. **Update** `data/jira_issues.json` so the stored ticket count stays current.
5. **Overview table** — monthly ticket counts by country.
6. **Excel** — ticket-level `Jira Issue ID -> Country`.

Country is resolved the same way for both outputs: the ticket's Ansprechpartner
asset (`customfield_10689`), escalating to Filiale (`customfield_10674`) and then
Zentrale (`customfield_10673`) — the first asset that carries a `Land` wins.


In [1]:
import os
import json
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv
import pytz

from jira_loader import fetch_jira_issues, dedupe_issues
from jira_country_export import resolve_countries

%load_ext autoreload
%autoreload 2
load_dotenv(override=True)

True

## Configuration

In [2]:
# How far back to pull, which projects, and where the artifacts live.
LOOKBACK_DAYS = 60
PROJECTS = [
    "SDAX",
    "SDIPR",
]  # both service desks are pulled fresh (brings in the current month)
MAX_ISSUES = 100000

STORE_PATH = "data/jira_issues.json"  # stored tickets (updated in place)
IPRO_PATH = "data/issues_ipro.json"  # SDIPR tickets to fold in (historical, pre-window)
OVERVIEW_CSV = "data/monthly_country_counts.csv"  # combined overview table
COUNTS_AX_CSV = (
    "data/monthly_country_counts_ax.csv"  # SDAX,   wide matrix (month_year x country)
)
COUNTS_IPRO_CSV = "data/monthly_country_counts_ipro.csv"  # SDIPR,  long (month_year, country, project, count)
EXCEL_PATH = "data/issue_country.xlsx"  # ticket-level knowledge

## 1. Load previously stored tickets

In [3]:
if os.path.exists(STORE_PATH):
    with open(STORE_PATH) as f:
        previous_issues = json.load(f)
else:
    previous_issues = []
print(f"Loaded {len(previous_issues)} previously stored tickets from {STORE_PATH}")

Loaded 62689 previously stored tickets from data/jira_issues.json


## 2. Pull the last `LOOKBACK_DAYS` days of tickets

`save_path=None` makes the pull return the tickets **without** overwriting the
store, so the existing data is safe until we merge and write it back ourselves.

In [4]:
tz = pytz.UTC
end_dt = datetime.now(tz)
start_dt = end_dt - timedelta(days=LOOKBACK_DAYS)
print(f"Pulling tickets created between {start_dt:%Y-%m-%d} and {end_dt:%Y-%m-%d}")

new_issues = []
for project in PROJECTS:
    pulled = fetch_jira_issues(
        start_dt, end_dt, max_issues=MAX_ISSUES, project=project, save_path=None
    )
    print(f"  {project}: {len(pulled)} tickets pulled")
    new_issues.extend(pulled)
print(f"Total newly pulled: {len(new_issues)}")

Pulling tickets created between 2026-06-25 and 2026-08-24
Fetching issues with JQL: project = SDAX AND created >= '2026-06-25 09:15' AND created <= '2026-08-24 09:15' ORDER BY created DESC
Failed to fetch asset 9926cb30-3f07-4fb2-9c83-aa4fc551c721:230386: 403 Client Error: Forbidden for url: https://api.atlassian.com/ex/jira/8a3828c5-f874-43ce-9367-3d9b73c02832/jsm/assets/workspace/8a799a44-1189-445f-9b88-56372496d3f0/v1/object/230386 — body: {"errorMessages":["Sie sind nicht berechtigt, diese Aktion durchzuführen."],"errors":{}}
Failed to fetch asset 9926cb30-3f07-4fb2-9c83-aa4fc551c721:222987: 403 Client Error: Forbidden for url: https://api.atlassian.com/ex/jira/8a3828c5-f874-43ce-9367-3d9b73c02832/jsm/assets/workspace/8a799a44-1189-445f-9b88-56372496d3f0/v1/object/222987 — body: {"errorMessages":["Sie sind nicht berechtigt, diese Aktion durchzuführen."],"errors":{}}
Failed to fetch asset 9926cb30-3f07-4fb2-9c83-aa4fc551c721:237894: 403 Client Error: Forbidden for url: https://api.a

OSError: Could not find a suitable TLS CA certificate bundle, invalid path: /Users/ovm/github/EVEX/evex_jira_ui/.venv/lib/python3.14/site-packages/certifi/cacert.pem

## 2b. Add the SDIPR (IPRO) tickets from `data/issues_ipro.json`

The main store is almost entirely SDAX — the bulk of the SDIPR tickets live in
`data/issues_ipro.json` and are missing from it. Load them here so they are
folded into the joined list *before* the deduplication below. If you also add
`"SDIPR"` to `PROJECTS` above, the fresh pull will supply the newest versions
and the dedup step will prefer them over these stored ones.

In [ ]:
if os.path.exists(IPRO_PATH):
    with open(IPRO_PATH) as f:
        ipro_issues = json.load(f)
    print(f"Loaded {len(ipro_issues)} SDIPR tickets from {IPRO_PATH}")
else:
    ipro_issues = []
    print(f"{IPRO_PATH} not found -- skipping SDIPR import")

## 3. Merge new tickets into the store by Ticket Key (deduplicated)

`dedupe_issues` guarantees every `key` is unique across **store + SDIPR file +
new pull**. When the same key appears in more than one source, the **more recent
push wins** (newest `fields.updated`, tie-broken by the later record). The order
`previous + ipro + new` means a freshly pulled ticket beats both the stored copy
and the SDIPR-file copy. Any pre-existing duplicates in the store are collapsed
here too.

In [ ]:
prev_keys = {issue["key"] for issue in previous_issues}
ipro_keys = {issue["key"] for issue in ipro_issues}
new_keys = {issue["key"] for issue in new_issues}

# Dedupe by key across store + SDIPR file + new pull; the more recent push wins.
merged_issues = dedupe_issues(previous_issues + ipro_issues + new_issues)

print(
    f"Previously stored      : {len(previous_issues)} records, {len(prev_keys)} unique keys"
)
print(f"SDIPR file added (new) : {len(ipro_keys - prev_keys)}")
print(f"Pull updated (existing): {len(new_keys & (prev_keys | ipro_keys))}")
print(f"Pull added (new)       : {len(new_keys - prev_keys - ipro_keys)}")
print(f"Total after merge      : {len(merged_issues)}")

# Hard guarantee: no duplicate keys remain.
assert len({i["key"] for i in merged_issues}) == len(merged_issues)

## 4. Update the stored tickets on disk

In [ ]:
with open(STORE_PATH, "w") as f:
    json.dump(merged_issues, f, indent=4, ensure_ascii=False)
print(f"Wrote {len(merged_issues)} tickets to {STORE_PATH}")

## 5. Resolve the country for every ticket

Ansprechpartner (`customfield_10689`) -> Filiale (`customfield_10674`) ->
Zentrale (`customfield_10673`). This warms the asset cache concurrently and can
take a couple of minutes on the first run (the Assets API rate-limits, so it
backs off and retries automatically).

In [ ]:
countries = resolve_countries(merged_issues)  # {ticket key: country}
resolved = sum(1 for v in countries.values() if v)
print(
    f"Country resolved for {resolved}/{len(countries)} tickets "
    f"({len(countries) - resolved} without a country)"
)

## 6. Assemble a per-ticket frame (key, created month, country)

In [ ]:
rows = []
for issue in merged_issues:
    rows.append(
        {
            "Jira Issue ID": issue["key"],
            "created": issue.get("fields", {}).get("created"),
            "Country": countries.get(issue["key"]),
        }
    )
df = pd.DataFrame(rows)
df["created"] = pd.to_datetime(df["created"], errors="coerce", utc=True)
df["month_year"] = df["created"].dt.strftime("%Y-%m")
df.head()

## 7. Overview table — monthly ticket counts by country

In [ ]:
overview = (
    df.assign(Country=df["Country"].fillna("No Country"))
    .groupby(["month_year", "Country"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)
overview.to_csv(OVERVIEW_CSV)
print(
    f"Wrote overview table ({overview.shape[0]} months x {overview.shape[1]} countries) "
    f"to {OVERVIEW_CSV}"
)
overview

## 7b. Monthly counts split per service desk

Regenerates the two per-desk files from the full merged store, so every month
present is included (e.g. the current month once it has been pulled). Both use
the same **wide matrix** layout — `month_year` plus one column per country:

* **SDAX**  -> `monthly_country_counts_ax.csv`
* **SDIPR** -> `monthly_country_counts_ipro.csv`

The `project` is taken from the ticket key prefix, and country uses the same
Ansprechpartner->Filiale->Zentrale resolution as everything else in this
notebook (so totals may differ slightly from older `customer_country`-based files).

In [ ]:
df_counts = df.assign(
    Country=df["Country"].fillna("No Country"),
    project=df["Jira Issue ID"].str.split("-").str[0],
)


def monthly_country_matrix(sub):
    """month_year x country wide matrix of ticket counts."""
    m = (
        sub.groupby(["month_year", "Country"])
        .size()
        .unstack(fill_value=0)
        .sort_index()
        .reset_index()
    )
    m.columns.name = None
    return m


# SDAX -> wide matrix (month_year x country)
ax_matrix = monthly_country_matrix(df_counts[df_counts["project"] == "SDAX"])
ax_matrix.to_csv(COUNTS_AX_CSV, index=False)
print(
    f"Wrote {COUNTS_AX_CSV}: {ax_matrix.shape[0]} months "
    f"({ax_matrix['month_year'].min()} .. {ax_matrix['month_year'].max()})"
)

# SDIPR -> same wide matrix layout
ipro_matrix = monthly_country_matrix(df_counts[df_counts["project"] == "SDIPR"])
ipro_matrix.to_csv(COUNTS_IPRO_CSV, index=False)
print(
    f"Wrote {COUNTS_IPRO_CSV}: {ipro_matrix.shape[0]} months "
    f"({ipro_matrix['month_year'].min()} .. {ipro_matrix['month_year'].max()})"
)

ipro_matrix.tail(3)

## 8. Excel — ticket-level knowledge (`Jira Issue ID` | `Country`)

In [ ]:
df_excel = df[["Jira Issue ID", "Country", "month_year"]]
df_excel.to_excel(EXCEL_PATH, index=False, sheet_name="Issue Country")
print(f"Wrote {len(df_excel)} rows to {EXCEL_PATH}")
df_excel["Country"].value_counts(dropna=False)